---

### Bloque 0: Preparación y Parámetros

Primero, define las fechas de corte para que los splits de comunas coincidan exactamente con los de las regiones.

```python
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. Definir los límites temporales para el split (REEMPLAZA CON TUS FECHAS)
# Deben ser exactamente los mismos que usaste para las regiones
FECHA_FIN_TRAIN = '2021-12-31 23:00:00'
FECHA_FIN_VAL = '2022-12-31 23:00:00'

# Lista de sectores
sectores = ['R', 'C', 'P', 'I', 'T']

```

In [1]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import MinMaxScaler
import os
import joblib

# Años de conjuntos de datos
train_years=[2018, 2019]
val_years=[2020]
test_years=[2021]

# Lista de sectores
sectores = ["R", "C", "P", "I", "T"]

# Cargar temperatura regional
df_temp_reg = pd.read_parquet("../data/interim/temperatura_regional_lagged.parquet", engine="pyarrow")


---

### Bloque 1: Actualizar los Datos Regionales

Lo más fácil para empezar y tacharlo de la lista. Cargamos los datos regionales ya listos y les agregamos la bandera espacial.

```python
# Cargar tus datos regionales (ciegos y metadatos)
# df_train_reg = pd.read_parquet(...)
# df_val_reg = pd.read_parquet(...)
# df_test_reg = pd.read_parquet(...)

# Función rápida para agregar la columna a todos
def add_comuna_flag(df_list):
    for df in df_list:
        df['is_comuna'] = 0 # 0 = Región
    return df_list

[df_train_reg, df_val_reg, df_test_reg] = add_comuna_flag([df_train_reg, df_val_reg, df_test_reg])
# Haz lo mismo para los DataFrames con metadatos regionales

print("-> Datos regionales actualizados con is_comuna = 0")

```


In [3]:
# Actualización de datos regionales
# Cargar datos con metadatos 
df_train_md_reg = pd.read_parquet("../data/processed/train_md_caso_3.parquet", engine="pyarrow")
df_val_md_reg = pd.read_parquet("../data/processed/val_md_caso_3.parquet", engine="pyarrow")
df_test_md_reg = pd.read_parquet("../data/processed/test_md_caso_3.parquet", engine="pyarrow")

# Función para agregar una columna a los datos
def add_comuna_flag(df_list, is_comuna=True): 
    for df in df_list: 
        df["is_comuna"] = 0 if not is_comuna else 1
    return df_list

df_reg = [df_train_md_reg, df_val_md_reg, df_test_md_reg]

[df_train_md_reg, df_val_md_reg, df_test_md_reg] = add_comuna_flag(df_reg, is_comuna=False)

# 1. Definir las columnas de temperatura exactas que vamos a reemplazar
cols_temp = ['temperatura', 'temp_t - 1', 'temp_t - 2', 'temp_t - 3', 
             'temp_t - 4', 'temp_t - 5', 'temp_t - 6', 'temp_t - 7']

# 2. Función para reemplazar las temperaturas escaladas por las reales (brutas)
def replace_scaled_temps(df_target, df_source, merge_keys, cols_to_replace):
    # A. Eliminar las columnas de temperatura escaladas (si existen) para no duplicarlas
    df_target_clean = df_target.drop(columns=[c for c in cols_to_replace if c in df_target.columns])
    
    # B. Seleccionar del dataset fuente SOLO las llaves y las temperaturas
    df_source_subset = df_source[merge_keys + cols_to_replace]
    
    # C. Realizar el cruce (Left merge para no perder ninguna fila de nuestros sets originales)
    df_merged = pd.merge(df_target_clean, df_source_subset, on=merge_keys, how='left')
    
    return df_merged

# 3. Aplicar el cruce a tus tres datasets regionales
print("Reemplazando temperaturas escaladas por datos crudos (brutos)...")
llaves_cruce = ['region', 'fecha_hora']

df_train_md_reg = replace_scaled_temps(df_train_md_reg, df_temp_reg, llaves_cruce, cols_temp)
df_val_md_reg   = replace_scaled_temps(df_val_md_reg, df_temp_reg, llaves_cruce, cols_temp)
df_test_md_reg  = replace_scaled_temps(df_test_md_reg, df_temp_reg, llaves_cruce, cols_temp)

# 4. Verificación rápida de sanidad (Sanity Check)
print(f"-> Train shape tras cruce: {df_train_md_reg.shape}")
print(f"-> Valores nulos en temperatura (Train): {df_train_md_reg['temperatura'].isna().sum()}")

# Imprimimos unas filas para comprobar visualmente que volvemos a tener grados Celsius reales
print("\nMuestra de las temperaturas (ahora deberían estar en °C reales):")
print(df_train_md_reg[['region', 'fecha_hora'] + cols_temp].head(3))

Reemplazando temperaturas escaladas por datos crudos (brutos)...
-> Train shape tras cruce: (174034, 31)
-> Valores nulos en temperatura (Train): 0

Muestra de las temperaturas (ahora deberían estar en °C reales):
        region          fecha_hora  temperatura  temp_t - 1  temp_t - 2  \
0  ANTOFAGASTA 2018-08-01 00:00:00    15.225098   16.353546   17.541382   
1  ANTOFAGASTA 2018-08-01 01:00:00    14.094360   15.225098   16.353546   
2  ANTOFAGASTA 2018-08-01 02:00:00    13.206116   14.094360   15.225098   

   temp_t - 3  temp_t - 4  temp_t - 5  temp_t - 6  temp_t - 7  
0   18.793640   19.782166   20.261780   20.228943   19.674835  
1   17.541382   18.793640   19.782166   20.261780   20.228943  
2   16.353546   17.541382   18.793640   19.782166   20.261780  



---

### Bloque 2: Procesar Shares y Ratios Comunales (MENSUAL)

Aquí calculamos las proporciones basándonos en tu base mensual.

```python
# Supongamos que tienes un DataFrame df_mensual con: 
# ['año', 'mes', 'comuna', 'demanda_comuna_mes', 'demanda_nacional_mes', 'consumo_R', 'consumo_C', ...]

# 1. Ratio Intensidad Comuna / Nacional (Mensual)
df_mensual['ratio_zona_nacional'] = df_mensual['demanda_comuna_mes'] / df_mensual['demanda_nacional_mes']

# 2. Shares Sectoriales / Comuna (Mensual)
for sector in sectores:
    col_consumo = f'consumo_{sector}'
    col_share = f'share_{sector}'
    # Evitar divisiones por cero
    df_mensual[col_share] = np.where(
        df_mensual['demanda_comuna_mes'] > 0,
        df_mensual[col_consumo] / df_mensual['demanda_comuna_mes'],
        0.0
    )

# Quédate solo con las columnas útiles para el cruce
cols_merge_mensual = ['año', 'mes', 'comuna', 'ratio_zona_nacional'] + [f'share_{s}' for s in sectores]
df_shares_mensuales = df_mensual[cols_merge_mensual].copy()

# Verifica que los shares sumen 1 (o muy cerca de 1)
print("Suma promedio de shares por fila:", df_shares_mensuales[[f'share_{s}' for s in sectores]].sum(axis=1).mean())

```


In [4]:
# Cargamos datos de demanda comunal y los shares
df_consumos_comunales = pd.read_parquet("../data/interim/shares_comunales.parquet", engine="pyarrow")
df_demanda_cen = pd.read_csv("../data/raw/demanda_comunal_horaria.csv")

# 1. Asegurar formato datetime y extraer 'año' y 'mes' para los cruces
# Para la demanda horaria del CEN:
df_demanda_cen['valid_time'] = pd.to_datetime(df_demanda_cen['valid_time'])
df_demanda_cen['año'] = df_demanda_cen['valid_time'].dt.year
df_demanda_cen['mes'] = df_demanda_cen['valid_time'].dt.month

# Para los consumos mensuales comunales:
df_consumos_comunales['año_ms'] = pd.to_datetime(df_consumos_comunales['año_ms'])
# El 'año' ya viene, pero nos aseguramos de tener el 'mes'
df_consumos_comunales['mes'] = df_consumos_comunales['año_ms'].dt.month 

# 2. Calcular la demanda NACIONAL mensual (desde df_demanda_cen)
# Sumamos la demanda de todas las comunas agrupando por año y mes
df_nacional_mensual = df_demanda_cen.groupby(['año', 'mes'])['demanda_mwh'].sum().reset_index()
df_nacional_mensual.rename(columns={'demanda_mwh': 'demanda_nacional_mes_mwh'}, inplace=True)

# 3. Cruzar los consumos comunales con el total nacional mensual
df_shares_mensuales = df_consumos_comunales.merge(
    df_nacional_mensual, 
    on=['año', 'mes'], 
    how='left'
)

# 4. Calcular el share espacial (Intensidad Comunal / Nacional)
# IMPORTANTE: La llamamos 'region_share' para homologar con df_test_reg
df_shares_mensuales['region_share'] = df_shares_mensuales['consumo_total_mes_MWh'] / df_shares_mensuales['demanda_nacional_mes_mwh']

# 5. Calcular los shares sectoriales mensuales
for sector in sectores:
    col_consumo = f'consumo_{sector}_MWh'
    col_share = f'share_{sector}'
    
    # Usamos np.where para evitar división por cero (por si alguna comuna/mes tiene 0 MWh)
    df_shares_mensuales[col_share] = np.where(
        df_shares_mensuales['consumo_total_mes_MWh'] > 0,
        df_shares_mensuales[col_consumo] / df_shares_mensuales['consumo_total_mes_MWh'],
        0.0
    )

# 6. Seleccionar SOLO las columnas unificadas y las llaves ('año', 'mes', 'comuna')
# El orden de los shares de los sectores lo dejé igual al que me pasaste de df_test_reg
cols_finales_shares = [
    'año', 'mes', 'comuna', # Llaves para cruzar en el Bloque 5
    'region_share',         # Nuestro ratio de intensidad comunal
    'share_I', 'share_R', 'share_C', 'share_P', 'share_T' # Shares sectoriales
]

df_shares_mensuales_limpio = df_shares_mensuales[cols_finales_shares].copy()
df_shares_mensuales_limpio = df_shares_mensuales_limpio.dropna(subset="region_share")

# ---- Sanity Check (Verificación) ----
print("=== Vista del DataFrame de Shares Mensuales Comunales ===")
print(df_shares_mensuales_limpio.head())
print("\nComprobación de que los shares sectoriales suman 1 (o aprox):")
# Suma de shares de una fila al azar
print(df_shares_mensuales_limpio[['share_I', 'share_R', 'share_C', 'share_P', 'share_T']].sum(axis=1).head())


=== Vista del DataFrame de Shares Mensuales Comunales ===
       año  mes                comuna  region_share   share_I   share_R  \
1516  2018    8           ANTOFAGASTA      0.156015  0.908140  0.042917   
1517  2018    8                CALAMA      0.084997  0.935679  0.037354   
1518  2018    8           MARIA ELENA      0.000000  0.000000  0.000000   
1519  2018    8            MEJILLONES      0.037116  0.977824  0.005851   
1520  2018    8  SAN PEDRO DE ATACAMA      0.010137  1.000000  0.000000   

       share_C   share_P  share_T  
1516  0.025234  0.023709      0.0  
1517  0.014274  0.012693      0.0  
1518  0.000000  0.000000      0.0  
1519  0.010602  0.005723      0.0  
1520  0.000000  0.000000      0.0  

Comprobación de que los shares sectoriales suman 1 (o aprox):
1516    1.0
1517    1.0
1518    0.0
1519    1.0
1520    1.0
dtype: float64



---

### Bloque 3: Estandarización de Demanda Comunal (ANUAL)

Recuerda que la estandarización (mu y sigma) debe ser estrictamente anual para no matar la estacionalidad del modelo.

```python
# Supongamos que df_demanda_horaria tiene: ['año', 'mes', 'fecha_hora', 'comuna', 'demanda_mwh']

# 1. Calcular mu y sigma ANUAL por comuna
df_stats_anual = df_demanda_horaria.groupby(['año', 'comuna'])['demanda_mwh'].agg(
    mu_total='mean',
    sigma_total='std'
).reset_index()

# 2. (Opcional pero recomendado) Calcular mu y sigma SIN cada sector
# Para esto, necesitas restar el consumo del sector a la demanda horaria y luego sacar mean/std.
# Si lo haces por escenarios puros en Kusumoto, deberás calcular las demandas horarias sin el sector "X" 
# basándote en los shares mensuales antes de hacer este groupby.

# 3. Unir stats al dataframe horario y estandarizar
df_target_comunal = df_demanda_horaria.merge(df_stats_anual, on=['año', 'comuna'], how='left')

# Llenar NaNs en sigma (por si hay varianza 0) con un número pequeño
df_target_comunal['sigma_total'] = df_target_comunal['sigma_total'].replace(0, 1e-6).fillna(1e-6)

# Estandarización
df_target_comunal['target_value'] = (df_target_comunal['demanda_mwh'] - df_target_comunal['mu_total']) / df_target_comunal['sigma_total']

print("-> Demanda estandarizada anualmente.")

```


In [5]:
print("Iniciando Bloque 3: Cálculo de mu, sigma y estandarización...")

# Filtros previos
# Forzar que el 'año' sea un número entero (int) en ambas bases
df_consumos_comunales['año'] = df_consumos_comunales['año'].astype(int)
df_demanda_cen['año'] = df_demanda_cen['año'].astype(int)

# Forzar que las comunas sean strings, en mayúsculas y sin espacios al inicio/final
df_consumos_comunales['comuna'] = df_consumos_comunales['comuna'].astype(str).str.upper().str.strip()
df_demanda_cen['comuna'] = df_demanda_cen['comuna'].astype(str).str.upper().str.strip()

# 1. Calcular las fracciones sectoriales ANUALES por comuna
# Agrupamos los consumos mensuales para obtener los totales del año
cols_consumo = [f'consumo_{s}_MWh' for s in sectores] + ['consumo_total_mes_MWh']
df_consumos_anuales = df_consumos_comunales.groupby(['año', 'comuna'])[cols_consumo].sum().reset_index()

# Renombrar la columna total para evitar confusiones
df_consumos_anuales.rename(columns={'consumo_total_mes_MWh': 'consumo_total_anual_MWh'}, inplace=True)

# Calcular la fracción anual por sector
for sector in sectores:
    col_consumo = f'consumo_{sector}_MWh'
    col_fraccion = f'fraccion_anual_{sector}'
    
    df_consumos_anuales[col_fraccion] = np.where(
        df_consumos_anuales['consumo_total_anual_MWh'] > 0,
        df_consumos_anuales[col_consumo] / df_consumos_anuales['consumo_total_anual_MWh'],
        0.0
    )

# 2. Calcular mu_total y sigma_total ANUAL por comuna a partir de la demanda horaria
df_stats = df_demanda_cen.groupby(['año', 'comuna'])['demanda_mwh'].agg(
    mu_total='mean',
    sigma_total='std'
).reset_index()

# Llenar NaNs en sigma (casos raros de varianza 0 o 1 solo dato) para evitar divisiones por cero
df_stats['sigma_total'] = df_stats['sigma_total'].replace(0, 1e-6).fillna(1e-6)

# 3. Cruzar Stats Base con Fracciones Anuales para obtener escenarios "Sin Sector"
df_stats = df_stats.merge(
    df_consumos_anuales[['año', 'comuna'] + [f'fraccion_anual_{s}' for s in sectores]], 
    on=['año', 'comuna'], 
    how='left'
)

# Rellenar posibles NaNs en las fracciones con 0.0 (por si alguna comuna no reportó consumos)
df_stats.fillna({f'fraccion_anual_{s}': 0.0 for s in sectores}, inplace=True)

# Verificación rápida del merge
exito_merge = df_stats['fraccion_anual_R'].sum()
print(f"Suma de fracciones residenciales en df_stats tras el merge: {exito_merge}")

# Calcular mu y sigma "sin sector"
for sector in sectores:
    fraccion = df_stats[f'fraccion_anual_{sector}']
    df_stats[f'mu_sin_{sector}'] = df_stats['mu_total'] * (1 - fraccion)
    df_stats[f'sigma_sin_{sector}'] = df_stats['sigma_total'] * (1 - fraccion)

# Limpiar las columnas de fracciones anuales ya que no las necesitamos en df_stats final
cols_drop = [f'fraccion_anual_{s}' for s in sectores]
df_stats.drop(columns=cols_drop, inplace=True)

# 4. (Opcional pero recomendado) Guardar df_stats para la fase de Inferencia/Desagregación
OUT_DIR = "../data/models/std_scale" # Asegúrate de definir tu ruta
df_stats.to_csv(os.path.join(OUT_DIR, "parametros_estandarizacion_comunal.csv"), index=False)
print("-> DataFrame df_stats generado y listo para guardar.")

# 5. Cruzar mu_total y sigma_total con la base horaria original para estandarizar
df_target_comunal = df_demanda_cen.merge(
    df_stats[['año', 'comuna', 'mu_total', 'sigma_total']], 
    on=['año', 'comuna'], 
    how='left'
)

# 6. Estandarizar la demanda horaria
# IMPORTANTE: La llamamos 'target_scaled' para que coincida exactamente con df_test_reg
df_target_comunal['target_scaled'] = (df_target_comunal['demanda_mwh'] - df_target_comunal['mu_total']) / df_target_comunal['sigma_total']

# ---- Sanity Check (Verificación) ----
print("\n=== Vista de df_stats (Parámetros Anuales) ===")
print(df_stats.head())

print("\n=== Vista de df_target_comunal (Demanda Estandarizada) ===")
print(df_target_comunal[['valid_time', 'comuna', 'demanda_mwh', 'mu_total', 'sigma_total', 'target_scaled']].head())

Iniciando Bloque 3: Cálculo de mu, sigma y estandarización...
Suma de fracciones residenciales en df_stats tras el merge: 217.7774760271151
-> DataFrame df_stats generado y listo para guardar.

=== Vista de df_stats (Parámetros Anuales) ===
    año           comuna  mu_total  sigma_total  mu_sin_R  sigma_sin_R  \
0  2018        ALGARROBO  3.341352     0.829690  1.183447     0.293861   
1  2018            ALHUÉ  0.881714     1.786091  0.881714     1.786091   
2  2018  ALTO DEL CARMEN  1.736457     0.707980  1.289273     0.525657   
3  2018    ALTO HOSPICIO  4.941135     6.256497  1.934711     2.449744   
4  2018            ANCUD  7.080370     1.785093  3.798903     0.957774   

   mu_sin_C  sigma_sin_C  mu_sin_P  sigma_sin_P  mu_sin_I  sigma_sin_I  \
0  2.749628     0.682760  2.749628     0.682760  3.341352     0.829690   
1  0.881714     1.786091  0.881714     1.786091  0.881714     1.786091   
2  1.091820     0.445152  1.091820     0.445152  1.736457     0.707980   
3  3.704539     4.


---

### Bloque 4: Escalar Temperaturas Comunales

Las redes neuronales necesitan la temperatura escalada, idealmente entre 0 y 1.

```python
# df_temp_lags tiene las columnas de fecha, comuna y temp_lag_0 hasta temp_lag_7

cols_temp = [col for col in df_temp_lags.columns if 'temp' in col.lower()]

# Inicializar scaler
scaler_temp_comunal = MinMaxScaler()

# ¡OJO AQUI! Solo debes hacer .fit() con los datos que caerán en TRAIN
filtro_train = df_temp_lags['fecha_hora'] <= FECHA_FIN_TRAIN
scaler_temp_comunal.fit(df_temp_lags.loc[filtro_train, cols_temp])

# Transformar todo el dataset
df_temp_lags[cols_temp] = scaler_temp_comunal.transform(df_temp_lags[cols_temp])

print("-> Temperaturas comunales escaladas (ajustadas solo con Train).")

```

CORRECCIÓN: 

En este bloque, se hará la importación de las temperaturas comunales, dado que el escalamiento se tiene que hacer común con el de las temperaturas regionales. Así, dicha tarea se ejecutará en el bloque final de trabajo, luego de separar los datos de entrenamiento, validación y testeo en los intervalos temporales indicados en el bloque 1.

In [6]:
print("Iniciando Bloque 4: Preparación de lags de temperatura comunal...")

# 1. Cargar el dataset de temperaturas comunales con lags (asumiendo que ya lo cargaste)
df_temp_comunal_lag = pd.read_parquet("../../prototipo_2/data/processed/temperatura_comunal_lagged.parquet", engine="pyarrow")

# 2. Asegurar que las llaves de cruce tengan el formato exacto
df_temp_comunal_lag['fecha_hora'] = pd.to_datetime(df_temp_comunal_lag['fecha_hora'])
df_temp_comunal_lag['comuna'] = df_temp_comunal_lag['comuna'].astype(str).str.upper().str.strip()

# Identificamos las columnas de temperatura para tenerlas mapeadas
cols_temp = [col for col in df_temp_comunal_lag.columns if 'temp' in col.lower()]

print(f"-> Columnas de temperatura detectadas: {cols_temp}")
print("-> Temperaturas comunales listas para el cruce. (El MinMaxScaler se aplicará globalmente en el Bloque 6).")

# ---- Sanity Check ----
print("\n=== Vista de df_temp_comunal_lag ===")
print(df_temp_comunal_lag[['fecha_hora', 'comuna'] + cols_temp].head())

Iniciando Bloque 4: Preparación de lags de temperatura comunal...
-> Columnas de temperatura detectadas: ['temperatura', 'temp_t - 1', 'temp_t - 2', 'temp_t - 3', 'temp_t - 4', 'temp_t - 5', 'temp_t - 6', 'temp_t - 7']
-> Temperaturas comunales listas para el cruce. (El MinMaxScaler se aplicará globalmente en el Bloque 6).

=== Vista de df_temp_comunal_lag ===
           fecha_hora     comuna  temperatura  temp_t - 1  temp_t - 2  \
0 2017-01-01 07:00:00  ALGARROBO    15.459625   15.466217   15.406891   
1 2017-01-01 08:00:00  ALGARROBO    15.397369   15.459625   15.466217   
2 2017-01-01 09:00:00  ALGARROBO    15.302643   15.397369   15.459625   
3 2017-01-01 10:00:00  ALGARROBO    15.225006   15.302643   15.397369   
4 2017-01-01 11:00:00  ALGARROBO    15.949371   15.225006   15.302643   

   temp_t - 3  temp_t - 4  temp_t - 5  temp_t - 6  temp_t - 7  
0   15.500153   15.672028   15.876617   16.003570   16.261871  
1   15.406891   15.500153   15.672028   15.876617   16.003570  
2   15


---

### Bloque 5: Ensamblaje del Dataset Comunal y Split Temporal

Ahora unimos todas las piezas comunales y dividimos en train, val y test.

```python
# 1. Mega Merge Comunal
df_comunal_full = df_target_comunal.copy()

# Agregar lags de temperatura
df_comunal_full = df_comunal_full.merge(df_temp_lags, on=['fecha_hora', 'comuna'], how='left')

# Agregar variables de calendario (df_calendario)
# df_comunal_full = df_comunal_full.merge(df_calendario, on=['fecha_hora'], how='left')

# Agregar shares y ratios mensuales (Cruzar por año, mes y comuna)
df_comunal_full = df_comunal_full.merge(df_shares_mensuales, on=['año', 'mes', 'comuna'], how='left')

# 2. Agregar la bandera espacial
df_comunal_full['is_comuna'] = 1

# Limpieza (Drop NaNs generados por los shifts de los lags)
df_comunal_full = df_comunal_full.dropna().reset_index(drop=True)

# 3. Temporal Split
df_train_com = df_comunal_full[df_comunal_full['fecha_hora'] <= FECHA_FIN_TRAIN].copy()
df_val_com   = df_comunal_full[(df_comunal_full['fecha_hora'] > FECHA_FIN_TRAIN) & (df_comunal_full['fecha_hora'] <= FECHA_FIN_VAL)].copy()
df_test_com  = df_comunal_full[df_comunal_full['fecha_hora'] > FECHA_FIN_VAL].copy()

print(f"Splits Comunales: Train {len(df_train_com)} | Val {len(df_val_com)} | Test {len(df_test_com)}")

```


In [7]:

print("Iniciando Bloque 5: Súper Merge Comunal...")

df_calendario_comunal = pd.read_parquet("../../prototipo_2/data/interim/calendario_comunal_features.parquet", engine="pyarrow")

# 1. Homologación de Llaves (El paso más importante para evitar NaNs silenciosos)
# Renombrar 'valid_time' a 'fecha_hora' en el target
if 'valid_time' in df_target_comunal.columns:
    df_target_comunal.rename(columns={'valid_time': 'fecha_hora'}, inplace=True)

# Asegurar formato datetime en todas las bases
df_target_comunal['fecha_hora'] = pd.to_datetime(df_target_comunal['fecha_hora'])
df_temp_comunal_lag['fecha_hora'] = pd.to_datetime(df_temp_comunal_lag['fecha_hora'])
df_calendario_comunal['fecha_hora'] = pd.to_datetime(df_calendario_comunal['fecha_hora'])

# Asegurar que 'comuna' sea string limpio en todas las bases (como aprendimos en el Bloque 3)
bases = [df_target_comunal, df_temp_comunal_lag, df_calendario_comunal, df_shares_mensuales_limpio]
for df in bases:
    if 'comuna' in df.columns:
        df['comuna'] = df['comuna'].astype(str).str.upper().str.strip()

# 2. MEGA MERGE
# Partimos de la demanda estandarizada como base fundamental
df_comunal_full = df_target_comunal.copy()

# A. Unir Lags de Temperatura Comunal (Cruce por fecha_hora y comuna)
df_comunal_full = df_comunal_full.merge(
    df_temp_comunal_lag, 
    on=['fecha_hora', 'comuna'], 
    how='inner' # Usamos inner para descartar horas sin temperatura
)

# B. Unir Features de Calendario (Cruce por fecha_hora y comuna)
df_comunal_full = df_comunal_full.merge(
    df_calendario_comunal, 
    on=['fecha_hora', 'comuna'], 
    how='inner'
)

# C. Unir Shares Sectoriales y Ratios (Cruce por año, mes y comuna)
df_comunal_full = df_comunal_full.merge(
    df_shares_mensuales_limpio, 
    on=['año', 'mes', 'comuna'], 
    how='inner'
)

# 3. Agregar la bandera espacial
df_comunal_full['is_comuna'] = 1

# 4. Limpieza Final (Botar NaNs generados por los shifts de temperatura u otros)
filas_antes = len(df_comunal_full)
df_comunal_full = df_comunal_full.dropna().reset_index(drop=True)
filas_despues = len(df_comunal_full)

print(f"-> Súper Merge completado. Filas descartadas por NaNs: {filas_antes - filas_despues}")

# 5. Split Temporal
df_train_com = df_comunal_full[df_comunal_full['año'].isin(train_years)].copy()
df_val_com   = df_comunal_full[df_comunal_full['año'].isin(val_years)].copy()
df_test_com  = df_comunal_full[df_comunal_full['año'].isin(test_years)].copy()

# ---- Sanity Check ----
print("\n=== Resumen de Splits Comunales ===")
print(f"Train Comunal (Años {train_years}): {len(df_train_com)} filas")
print(f"Val Comunal   (Años {val_years}): {len(df_val_com)} filas")
print(f"Test Comunal  (Años {test_years}): {len(df_test_com)} filas")

print("\n=== Vista de Columnas Finales ===")
print(df_train_com.columns.tolist())

Iniciando Bloque 5: Súper Merge Comunal...
-> Súper Merge completado. Filas descartadas por NaNs: 0

=== Resumen de Splits Comunales ===
Train Comunal (Años [2018, 2019]): 1885186 filas
Val Comunal   (Años [2020]): 1414385 filas
Test Comunal  (Años [2021]): 1417146 filas

=== Vista de Columnas Finales ===
['fecha_hora', 'comuna', 'demanda_mwh', 'año', 'mes', 'mu_total', 'sigma_total', 'target_scaled', 'temperatura', 'temp_t - 1', 'temp_t - 2', 'temp_t - 3', 'temp_t - 4', 'temp_t - 5', 'temp_t - 6', 'temp_t - 7', 'region', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'is_working_day', 'is_holiday', 'is_weekend', 'region_share', 'share_I', 'share_R', 'share_C', 'share_P', 'share_T', 'is_comuna']



---

### Bloque 6: Fusión Final (Regiones + Comunas) y "Shuffle"

El paso final. Juntamos las tablas regionales (actualizadas en el Bloque 1) con las comunales, y las revolvemos para que el modelo aprenda a generalizar.

```python
# Asegúrate de que las columnas de los dataframes regionales y comunales se llamen EXACTAMENTE igual
# cols_finales = ['is_comuna', 'ratio_zona_nacional', 'share_R', ... 'temp_lag_0', ... 'target_value']

# Concatenar
train_global = pd.concat([df_train_reg[cols_finales], df_train_com[cols_finales]], ignore_index=True)
val_global   = pd.concat([df_val_reg[cols_finales],   df_val_com[cols_finales]], ignore_index=True)
test_global  = pd.concat([df_test_reg[cols_finales],  df_test_com[cols_finales]], ignore_index=True)

# ¡MUY IMPORTANTE! Barajar (Shuffle) los datos de entrenamiento
# Si no lo haces, la red neuronal verá primero puras regiones y luego puras comunas, y olvidará las regiones (Catastrophic Forgetting)
train_global = train_global.sample(frac=1, random_state=42).reset_index(drop=True)

print("¡Dataset Global Listo para Entrenar!")
print(f"Tamaño Train Global: {len(train_global)}")

```

In [10]:
print("Iniciando Bloque 6: Fusión Global, Re-escalamiento y Ajuste de Atributos...")

# 1. Ajuste de datasets Regionales
for df in [df_train_md_reg, df_val_md_reg, df_test_md_reg]:

    df['target_scaled'] = (df['demanda_mwh'] - df['mu_total']) / df['sigma_total']
    
    # A) Unificamos el identificador espacial
    if 'region' in df.columns:
        df['region_comuna'] = df['region']
        df.drop(columns=['region'], inplace=True)
        
    # B) Renombramos el share
    if 'region_share' in df.columns:
        df.rename(columns={'region_share': 'region_comuna_share'}, inplace=True)
        
    # C) Aseguramos el flag 
    df['is_comuna'] = 0.0

# 2. Ajuste de datasets Comunales
for df in [df_train_com, df_val_com, df_test_com]:
    
    # A) Unificamos el identificador espacial
    if 'comuna' in df.columns:
        df['region_comuna'] = df['comuna']
        
    # B) Eliminamos las columnas espaciales redundantes
    cols_to_drop = [c for c in ['region', 'comuna'] if c in df.columns]
    if cols_to_drop:
        df.drop(columns=cols_to_drop, inplace=True)
        
    # C) Renombramos el share
    if 'region_share' in df.columns:
        df.rename(columns={'region_share': 'region_comuna_share'}, inplace=True)
        
    # D) Aseguramos el flag
    df['is_comuna'] = 1.0

# 3. Fusión Global (Concatenación)
print("Concatenando datasets regionales y comunales...")
df_train_global = pd.concat([df_train_md_reg, df_train_com], ignore_index=True)
df_val_global = pd.concat([df_val_md_reg, df_val_com], ignore_index=True)
df_test_global = pd.concat([df_test_md_reg, df_test_com], ignore_index=True)

# 4. Nuevo Escalamiento Global (MinMaxScaler)
print("Aplicando MinMaxScaler global a las temperaturas combinadas...")
scaler_temp_global = MinMaxScaler()

# ¡CRÍTICO!: 'fit' en los datos de entrenamiento para evitar Data Leakage
df_train_global[cols_temp] = scaler_temp_global.fit_transform(df_train_global[cols_temp])

# Aplicamos 'transform' directo en Validación y Testeo
df_val_global[cols_temp] = scaler_temp_global.transform(df_val_global[cols_temp])
df_test_global[cols_temp] = scaler_temp_global.transform(df_test_global[cols_temp])

# Guardamos el nuevo scaler global, ya que lo necesitarás en el futuro (Inferencia/Desagregación)
joblib.dump(scaler_temp_global, "../models/scaler_temp_global.pkl")

# 5. Mezclado Aleatorio (Shuffling) - ¡CORREGIDO!
print("Generando variable mezclada exclusivamente para el set de Entrenamiento...")
# Creamos variables separadas y SOLO mezclamos Train. 
# Val y Test se mantienen en su orden cronológico original para facilitar gráficos posteriores.
df_train_shuffled = df_train_global.sample(frac=1, random_state=42).reset_index(drop=True)

# (Opcional) Puedes ordenar cronológicamente Validación y Testeo por si se mezclaron al concatenar
df_val_global = df_val_global.sort_values(by=['region_comuna', 'fecha_hora']).reset_index(drop=True)
df_test_global = df_test_global.sort_values(by=['region_comuna', 'fecha_hora']).reset_index(drop=True)

print(f"\n-> Train Global Original (Ordenado): {df_train_global.shape}")
print(f"-> Train Global Shuffled (Mezclado): {df_train_shuffled.shape}")
print(f"-> Val Global (Ordenado):   {df_val_global.shape}")
print(f"-> Test Global (Ordenado):  {df_test_global.shape}")

Iniciando Bloque 6: Fusión Global, Re-escalamiento y Ajuste de Atributos...
Concatenando datasets regionales y comunales...
Aplicando MinMaxScaler global a las temperaturas combinadas...
Generando variable mezclada exclusivamente para el set de Entrenamiento...

-> Train Global Original (Ordenado): (2059220, 33)
-> Train Global Shuffled (Mezclado): (2059220, 33)
-> Val Global (Ordenado):   (1537361, 33)
-> Test Global (Ordenado):  (1539786, 33)


In [ ]:
# Si el scaler fue entrenado (fit), DEBE tener guardados los valores mínimos y máximos.
try:
    print("Mínimos aprendidos:", scaler_temp_global.data_min_)
    print("Máximos aprendidos:", scaler_temp_global.data_max_)
    print("¡El scaler SÍ está entrenado!")
except AttributeError:
    print("ERROR: El scaler está VACÍO. No tiene data_min_. Nunca se le hizo 'fit'.")

Mínimos aprendidos: [-18.033173 -18.033173 -18.033173 -18.033173 -18.033173 -18.033173
 -18.033173 -18.033173]
Máximos aprendidos: [37.176178 37.176178 37.176178 37.176178 37.176178 37.176178 37.176178
 37.176178]
¡El scaler SÍ está entrenado! El problema es otro.


In [13]:
# Guardar datos
print("Iniciando Guardado Final de Datasets...")

# 1. Crear el directorio si no existe
out_dir = "../data/processed/ds_comunal/"
os.makedirs(out_dir, exist_ok=True)

# 2. Guardar datasets CON metadatos (Para futura inferencia, desescalamiento y gráficos)
print(f"Guardando metadatos en: {out_dir}")
df_train_global.to_parquet(os.path.join(out_dir, "train_metadata_global.parquet"), index=False)
df_val_global.to_parquet(os.path.join(out_dir, "val_metadata_global.parquet"), index=False)
df_test_global.to_parquet(os.path.join(out_dir, "test_metadata_global.parquet"), index=False)

# 3. Definir columnas a eliminar para hacer los datos "ciegos"
cols_to_drop = [
    'fecha_hora', 
    'region_bne', 
    'demanda_mwh', 
    'año', 
    'mu_total', 
    'sigma_total', 
    'region_comuna', 
    'mes'
]

# Función auxiliar para tirar columnas de forma segura
def make_blind(df, columns_to_remove):
    # Filtramos solo las columnas que realmente existen en el DataFrame actual
    cols_existentes = [c for c in columns_to_remove if c in df.columns]
    return df.drop(columns=cols_existentes)

# 4. Aplicar "ceguera" (Features + Target numérico estandarizado solamente)
# OJO: Usamos df_train_shuffled para el train, y los globales ordenados para val y test.
print("Eliminando metadatos para crear tensores de entrenamiento...")
df_train_ciego = make_blind(df_train_shuffled, cols_to_drop)
df_val_ciego   = make_blind(df_val_global, cols_to_drop)
df_test_ciego  = make_blind(df_test_global, cols_to_drop)

# 5. Guardar datasets "ciegos" listos para Keras / TensorFlow
print("Guardando datasets ciegos...")
df_train_ciego.to_parquet(os.path.join(out_dir, "train_ciego_global.parquet"), index=False)
df_val_ciego.to_parquet(os.path.join(out_dir, "val_ciego_global.parquet"), index=False)
df_test_ciego.to_parquet(os.path.join(out_dir, "test_ciego_global.parquet"), index=False)

print("\n¡Proceso Completado con Éxito!")
print(f"-> Train Ciego shape: {df_train_ciego.shape}")
print(f"-> Val Ciego shape:   {df_val_ciego.shape}")
print(f"-> Test Ciego shape:  {df_test_ciego.shape}")

Iniciando Guardado Final de Datasets...
Guardando metadatos en: ../data/processed/ds_comunal/
Eliminando metadatos para crear tensores de entrenamiento...
Guardando datasets ciegos...

¡Proceso Completado con Éxito!
-> Train Ciego shape: (2059220, 25)
-> Val Ciego shape:   (1537361, 25)
-> Test Ciego shape:  (1539786, 25)
